# Routing messages by intent

Uses the saved model to decide where each message goes, like in our diagram (Classifier / Forecast / APIs / LLM). The real handlers (database, exchange rates, Gemini) are in `ai-orchestrator/nlp_prototype/nlp_proto/orchestrator.py`; here we only check the routing.

In [1]:
import joblib
import pandas as pd

In [2]:
import re

AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize(text):
    text = str(text).translate(AR_DIGITS)
    text = re.sub("[إأآ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    text = re.sub(r"[\u064B-\u0652\u0640]", "", text)  # tashkeel + tatweel
    return text.lower().strip()

# "هصرف" (will spend) and "صرفت" (spent) look almost the same in char n-grams,
# so the model mixes up spending questions and forecasts. A future-tense verb fixes most of it.
FUTURE_RE = re.compile(
    r"(?:^|\s)(?:ه|هي|هن)(?:صرف|حوش|وفر|دفع|فضل|تبقي|كفي|خلص|قبض|يفضل|يتبقي|يكفي|يخلص)\w*"
    r"|\bwill i\b|\bam i going to\b|\bhow much will\b|\bha(?:sraf|7awesh|dfa3)\b|\bhayfdal\b",
    re.I,
)

In [3]:
intent_model = joblib.load("../../models/intent_tfidf.joblib")

UNSURE_BELOW = 0.35

def predict_intent(text):
    clean = normalize(text)
    probs = intent_model.predict_proba([clean])[0]
    best = probs.argmax()
    intent = intent_model.classes_[best]
    confidence = probs[best]

    if intent == "spending_query" and FUTURE_RE.search(clean):
        intent = "forecast"
    if confidence < UNSURE_BELOW:
        intent = "unsure"
    return intent, round(float(confidence), 2)

In [4]:
ROUTES = {
    "transaction":    "Transaction classifier -> save to database",
    "spending_query": "Query the user's transactions",
    "forecast":       "Forecast model",
    "market":         "Exchange-rate API / LLM with web search",
    "advice":         "LLM with the user's transactions",
    "manage":         "Delete / edit a saved transaction",
    "greeting":       "Fixed reply",
    "out_of_scope":   "Fixed reply (or LLM)",
    "unsure":         "Ask the user what they meant",
}

def route(text):
    intent, confidence = predict_intent(text)
    return {"text": text, "intent": intent, "confidence": confidence, "goes to": ROUTES[intent]}

In [5]:
messages = [
    "انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه",
    "قبضت المرتب ١٥٠٠٠",
    "اكبر مصروف عندي ايه",
    "How much did I spend on food this week?",
    "لو فضلت كده هحوش كام آخر السنة",
    "el euro b kam delwa2ty",
    "ايه هو سعر ال nissan magnite ؟",
    "اقدر اشتري عربية ماجنايت بمصاريفي دي؟",
    "غير ال 200 بتاعة الغدا ل 250",
    "صباح الخير",
    "مين هيكسب الماتش النهارده؟",
]
pd.DataFrame([route(m) for m in messages])

,text,intent,confidence,goes to
0,انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه,transaction,0.92,Transaction classifier -> save to database
1,قبضت المرتب ١٥٠٠٠,transaction,0.62,Transaction classifier -> save to database
2,اكبر مصروف عندي ايه,spending_query,0.67,Query the user's transactions
3,How much did I spend on food this week?,spending_query,0.96,Query the user's transactions
4,لو فضلت كده هحوش كام آخر السنة,forecast,0.97,Forecast model
5,el euro b kam delwa2ty,market,0.43,Exchange-rate API / LLM with web search
6,ايه هو سعر ال nissan magnite ؟,market,0.36,Exchange-rate API / LLM with web search
7,اقدر اشتري عربية ماجنايت بمصاريفي دي؟,advice,0.64,LLM with the user's transactions
8,غير ال 200 بتاعة الغدا ل 250,manage,0.97,Delete / edit a saved transaction
9,صباح الخير,greeting,0.98,Fixed reply


In [6]:
# try your own message
route("عايز اعرف صرفت كام على المواصلات")

{'text': 'عايز اعرف صرفت كام على المواصلات',
 'intent': 'spending_query',
 'confidence': 0.51,
 'goes to': "Query the user's transactions"}